In [129]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import ollama
import numpy as np

In [130]:
loader = PyPDFLoader('data/2312.10997v5.pdf')
loader = PyPDFLoader('data/10201979.pdf')
loader = PyPDFLoader('data/2509.23988v3.pdf')
loader = PyPDFLoader('data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf')
pages = loader.load()


### Text Cleaning

In [131]:
def cleanText(text): 
    cleaned_text = re.sub(r'\s+', ' ', text) # clears the spacess
    cleaned_text = cleaned_text.replace('-', " ")
    cleaned_text = cleaned_text.replace(' -', " ")
    cleaned_text = cleaned_text.replace('•', ";")
    cleaned_text = cleaned_text.replace("   ", " ")
    cleaned_text = cleaned_text.replace("__", "")
    return cleaned_text

In [132]:
for page in pages:
    page.page_content = cleanText(page.page_content)

In [133]:
pages[: 10]

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'keywords': '', 'title': '', 'subject': '', 'author': '', 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf', 'total_pages': 28, 'page': 0, 'page_label': '64087'}, page_content='Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025. Digital Object Identifier 10.1 109/ACCESS.2025.3556973 Exploring the Potential of Offline LLMs in Data Science: A Study on Code Generation for Data Analysis ANASTASIOS NIKOLAKOPOULOS 1, ANTONIOS LITKE 1, ALEXANDROS PSYCHAS 1, ELENI VERONI 2, AND THEODORA VARVARIGOU 1 1School of Electrical and Computer Engineering, Institute of

### Chunking
 - splitting text information into smaller sizes

In [134]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 400, chunk_overlap = 200)

In [135]:
chunks = splitter.split_documents(pages)

In [136]:
chunks[0:10]

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-04-15T21:43:08+05:30', 'moddate': '2025-04-17T21:53:13-04:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'keywords': '', 'title': '', 'subject': '', 'author': '', 'source': 'data/Exploring_the_Potential_of_Offline_LLMs_in_Data_Science_A_Study_on_Code_Generation_for_Data_Analysis.pdf', 'total_pages': 28, 'page': 0, 'page_label': '64087'}, page_content='Received 13 February 2025, accepted 28 March 2025, date of publication 1 April 2025, date of current version 18 April 2025. Digital Object Identifier 10.1 109/ACCESS.2025.3556973 Exploring the Potential of Offline LLMs in Data Science: A Study on Code Generation for Data Analysis ANASTASIOS NIKOLAKOPOULOS 1, ANTONIOS LITKE 1, ALEXANDROS PSYCHAS 1, ELENI VERONI 2, AND THEODORA VARVARIGOU 1 1School'),
 Document(metadata={'producer': 'pdfTeX-1.40.24',

In [137]:
len(chunks)

693

### Embeddings

In [138]:
EMBEDDING_MODEL = 'nomic-embed-text-v2-moe'
def createEmbedding(text):
    response = ollama.embed(
    model = EMBEDDING_MODEL,
    input = text ,
    )
    return response.embeddings

In [139]:
doc_embeddings = []

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding)

In [140]:
len(doc_embeddings)

693

In [141]:
user_query = input('Ask anything about the Document...')


In [142]:
print(user_query)   

what is RAG?


In [143]:
user_query_embedding  = createEmbedding(user_query)

In [144]:
print(user_query_embedding)

[[0.023178028, -0.07060313, -0.042724185, -0.048304144, 0.07726823, -0.051364467, -0.025396978, -0.0064781033, -0.06149865, 0.03320012, -0.024821337, -0.019715197, -0.0031334716, -0.004565455, -0.024641357, -0.039742157, 0.020846615, 0.036315583, 0.024593528, -0.033315763, -0.048175983, 0.035793204, -0.019613814, -0.029355453, 0.010953603, 0.02206241, -0.07356402, -0.014257479, 0.015768815, 0.05121068, -0.009017594, 0.01610734, -0.03750718, 0.030475775, -0.019803764, 0.010093743, -0.03404114, -0.00283465, 0.007215327, -0.020510126, 0.032875184, 0.041442465, 0.03854718, -0.033751648, -0.00851193, -0.05787889, -0.10991596, -0.0011362992, 0.016475163, -0.013944581, 0.005927594, -0.01603422, -0.005442964, -0.027397877, -0.059195183, -0.072963215, -0.020737123, -0.07350159, 0.05508017, -0.019074414, -0.03282287, -0.037449833, -0.040196482, 0.047414694, -0.039307196, 0.05214233, 0.052336812, 0.04501578, -0.022092493, 0.024414632, -0.06679565, -0.008909534, 0.00057197793, 0.0032487132, -0.003

In [145]:
above_zero_point_six = []

for index, doc_embedding in enumerate(doc_embeddings):
    score = cosine_similarity(doc_embedding, user_query_embedding)
    if score >= 0.6:
        result = (index, score)
        above_zero_point_six.append(result)

In [146]:
above_zero_point_six

[]

In [147]:
largest = max(above_zero_point_six, key=lambda x: x[1])
largest

ValueError: max() iterable argument is empty

In [148]:
response_to_client = chunks[largest[0]]
print(response_to_client.page_content)

NameError: name 'largest' is not defined

In [117]:
chunk[above_zero_point_six[2][0]].page_content

IndexError: list index out of range

### Evaluations

In [ ]:
a = np.array([1,2,3])
b = np.array([1,2,3])

In [ ]:
dot_sim = np.dot(a,b)

In [ ]:
cos_sim = np.dot(a,b) / (np.linalg.norm(a)* np.linalg.norm(b))

In [ ]:
euclidean_dist = np.linalg.norm(a - b)

In [ ]:
output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
output_cos = f" COSINE SIM_SCORE: {cos_sim} "
output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

print(f""" 
{output_dot} \n 
{output_cos} \n 
{output_eucl} \n 
""")

In [ ]:
def similarityChecks(text_one_embeding, text_two_embeding):
    # Dot Product Similarity
    dot_sim = np.dot(text_one_embeding, text_two_embeding)
    # Cosine Similarity
    cos_sim = np.dot(text_one_embeding, text_two_embeding) / (np.linalg.norm(text_one_embeding) * np.linalg.norm(text_two_embeding))
    # Euclidean Distance (Lower is better)
    euclidean_dist = np.linalg.norm(np.array(text_one_embeding) - np.array(text_two_embeding))   

    output_dot = f" DOT PRODUCT SIM_SCORE: {dot_sim} "
    output_cos = f" COSINE SIM_SCORE: {cos_sim} "
    output_eucl = f" EUCLIDEAN DISTANCE: {euclidean_dist} "

    print(f" {output_dot} \n {output_cos} \n {output_eucl} \n ")

In [ ]:
user_query = input("ask me anything...")